# Learned instance selection · Four-fold OOF

**EdgeAttNet / Component-quality modeling**

The 31-feature CatBoost selector is trained only on out-of-fold predictions. By default, this notebook skips retraining. In training mode, it reuses complete OOF pair outputs if present; otherwise it runs four pretrained fold models downloaded automatically from the public model repository, creates held-out probability maps, audits all 566 training images, and then refits CatBoost.


In [ ]:
# Locate the repository from its root or notebooks/ directory.
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p/'src'/'architecture.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook inside the extracted repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)


## 01 / Configuration

The original competition training images, annotations, and split tables are required for filter retraining. `PAIR_01_ROOT` / `PAIR_23_ROOT` may point to existing complete OOF outputs. When omitted, the notebook builds them from pretrained models (no OOF *training* required). Hugging Face artifacts must be public for anonymous committee use.

In [ ]:
from pathlib import Path
import subprocess,sys

TRAIN_FILTER_FROM_SCRATCH = False  # Default: use published CatBoost model
REPO_ROOT = Path.cwd()
DATA_ROOT = Path('/kaggle/input')
SPLIT_ROOT = REPO_ROOT/'data'/'splits'
PAIR_01_ROOT = None  # e.g. /kaggle/input/oof01/DINO_OOF_Folds01
PAIR_23_ROOT = None  # e.g. /kaggle/input/oof23/DINO_OOF_Folds23
HF_OOF_REPO = 'youssefElghanam191/edgeattnet-dinov3-convnext-base'
OOF_MODELS_ROOT = Path('/kaggle/input')   # Local first, Hugging Face fallback
DOWNLOAD_OOF_FROM_HF = True
GENERATE_MISSING_OOF_MAPS = True  # Infer with pretrained fold models, never train them
OOF_SESSION_HOURS = 9.5
OUTPUT_DIR = Path('/kaggle/working/learned_filter') if Path('/kaggle/working').exists() else REPO_ROOT/'outputs'/'learned_filter'

assert (REPO_ROOT/'scripts'/'fit_filter_oof.py').is_file(), 'Run from the repository root.'
print('Filter mode:', 'REFIT ON FOUR HELD-OUT FOLDS' if TRAIN_FILTER_FROM_SCRATCH else 'LOAD PUBLISHED PRETRAINED')

## 02 / Resolve OOF maps and refit

When retraining is enabled, both fold pairs are required. Missing outputs are generated using pretrained fold models with their original exclusion rosters. Their SHA-256 and provenance sidecars are audited before CatBoost sees a single probability map. The default pretrained path skips all four fold models.

In [ ]:
if TRAIN_FILTER_FROM_SCRATCH:
    generated = {}
    for pair, existing in [('01', PAIR_01_ROOT), ('23', PAIR_23_ROOT)]:
        if existing is not None:
            root = Path(existing)
            if not (root/'oof_training_manifest.json').is_file():
                raise FileNotFoundError(f'OOF pair {pair} has no manifest: {root}')
        else:
            if not GENERATE_MISSING_OOF_MAPS:
                raise FileNotFoundError(f'OOF pair {pair} missing. Enable GENERATE_MISSING_OOF_MAPS.')
            root = OUTPUT_DIR/f'DINO_OOF_Folds{pair}'
            cmd = [sys.executable, str(REPO_ROOT/'scripts'/'run_oof_pair.py'),
                   '--folds', pair, '--mode', 'pretrained',
                   '--models-root', str(OOF_MODELS_ROOT),
                   '--hf-repo', HF_OOF_REPO, '--splits-root', str(SPLIT_ROOT),
                   '--output', str(root), '--hours', str(OOF_SESSION_HOURS)]
            if not DOWNLOAD_OOF_FROM_HF:cmd.append('--no-hf-download')
            subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)
        generated[pair] = root
    PAIR_01_ROOT = generated['01']
    PAIR_23_ROOT = generated['23']
    index_dir=OUTPUT_DIR/'verified_oof'
    subprocess.run([sys.executable,str(REPO_ROOT/'scripts'/'audit_oof_coverage.py'),
                    '--pair-01',str(PAIR_01_ROOT),'--pair-23',str(PAIR_23_ROOT),
                    '--out',str(index_dir)],cwd=str(REPO_ROOT),check=True)
    subprocess.run([sys.executable,str(REPO_ROOT/'scripts'/'fit_filter_oof.py'),
                    '--dataset',str(DATA_ROOT),'--splits',str(SPLIT_ROOT),
                    '--oof',str(index_dir),'--out',str(OUTPUT_DIR/'model')],cwd=str(REPO_ROOT),check=True)
    print('Refitted CatBoost:',OUTPUT_DIR/'model'/'final_component_regressor_f.joblib')
else:
    from src.artifacts import resolve_artifacts, load_filter
    artifacts=resolve_artifacts(model_dir=None, hf_repo=HF_OOF_REPO, allow_download=True)
    model, choice=load_filter(artifacts['regressor'],artifacts['config'])
    print('Pretrained CatBoost loaded from:', artifacts['regressor'])
    print('Locked cutoff:',choice['cutoff'])


## 03 / Frozen decision rule

The submitted method used hysteresis **0.15 / 0.90**, minimum area **150**, and CatBoost depth **5**, **550 trees**, cutoff **0.25**. Re-fitting the same hyperparameters from independently generated OOF maps is valid source reproduction, but random seeds, numerical kernels and the fold models' initialization may prevent byte-identical CatBoost artifacts. Training the filter on the final model's own training predictions is not equivalent to OOF.